# Creating a Jupyter Kernel for this notebook

If you haven't done so yet, follow the instructions below to create a Kernel for this notebook with the right Python packages installed.

1) Run the following cell, reload the notebook and select your newly created kernel

In [ ]:
!python -m pip install ipykernel 
!python -m ipykernel install --user --name mango_python_client_training --display-name "ManGO - Python client training"

2) With your new kernel selected, run the following cell to install the needed python packages

In [ ]:
!which python # shows you which python (environment) you are using

In [ ]:
%pip install python-irodsclient
%pip install mango_auth
%pip install git+https://github.com/kuleuven/mango-mdschema.git@feature/read-irods-schema

Run this cell to see the packages installed in the current kernel's environment:

In [ ]:
%pip freeze

# Basic workflow with ManGO

<div class="alert alert-block alert-info">
<h3>Authentication with Python</h3>

You can authenticate to the Python client with the mango_auth library. 
The first time you do this (or when you switch to a different ManGO zone), you need to follow a url to authenticate. 
This can be done from your terminal, or inside the script.

Next times, a refresh token is used, so no human interaction is needed.

You can also authenticate via some other clients (iron and iCommands), whose authentication also works for Python scripts.
For instructions go to https://mango.vscentrum.be/ and in the tab of your zone, click on "How to connect".
</div>

In [ ]:
# Option 1: Authentication via the terminal
!bash -c "mango_auth <username> gbiomed gbiomed.irods.icts.kuleuven.be"

In [ ]:
# Option 2: Authencation in Python
from mango_auth import iinit
iinit('<username>', 'gbiomed', 'gbiomed.irods.icts.kuleuven.be')

## Setting up

Set up ManGO and load any other libraries you need.

In [ ]:
import base64 # to convert iRODS checksums
import glob
import hashlib # to compute local checksums
import os
from irods.session import iRODSSession # to communicate with ManGO
from irods.access import iRODSAccess # to set permissions
from mango_mdschema import Schema, ValidationError, ConversionError # to add structured metadata
from mango_mdschema.schema import get_mango_schema
import logging
logger = logging.getLogger("mango_mdschema") # to read the validation
logger.setLevel(logging.INFO)
try:
    env_file = os.environ['IRODS_ENVIRONMENT_FILE']
except KeyError:
    env_file = os.path.expanduser('~/.irods/irods_environment.json')

Since we are working interactively we will create an `irods.session.iRODSSession` object and then close it at the end of the notebook with `session.cleanup()`. If you were working on a script, you could run all your code inside a `with` statement.

In [ ]:
session = iRODSSession(irods_env_file=env_file)

In [ ]:
home_dir = "/gbiomed/home/training/Exercises"

## WRITE YOUR CODE BELOW THIS

Below, create a collection under /gbiomed/home/Exercises/output with your u-number (e.g. "u0000001") to store your output.

In [ ]:
output_dir = home_dir + "/output/" + "<username>" # it might be a different name
session.collections.create(output_dir)

Give the training project read access to your output folder, and make sure inheritance is on so they also get access on its future contents.

In [ ]:
from irods.access import iRODSAccess

session.acls.set(iRODSAccess('read',  output_dir, 'training'))
session.acls.set(iRODSAccess('inherit', output_dir))

If you want, you can check whether the permissions and inheritance have been set correctly

In [ ]:
coll = session.collections.get(output_dir)
print(session.acls.get(coll))
print("Inheritance:", coll.inheritance)

In the cell below, query for all data objects in the project annotated with the `fastq` Schema and for which the "organism" is "mouse".

In [ ]:
# importing necessary classes
from irods.models import Collection, DataObject, DataObjectMeta
from irods.column import Criterion

my_files = session.query(Collection.name, DataObject.name).filter(
    Criterion('like', Collection.name, home_dir + '%')).filter(
    Criterion('=', DataObjectMeta.name, 'mgs.fastq.organism')).filter(
    Criterion('=', DataObjectMeta.value, 'mouse')
    )
paths = [(item[Collection.name], item[DataObject.name]) for item in my_files]
print(paths)


Download all data objects returned by the query

In [ ]:
for collection_name, filename in paths:
    session.data_objects.get(f"{collection_name}/{filename}", filename)

Now that you have downloaded your data, run your analysis, and upload a detailed report to your personal output collection
(If you just upload an empty README file, that's also okay, I won't tell anyone ;) )

In [ ]:
output_filename = 'README.txt'
with open('README.txt', 'w') as file:
    file.write("Yes, that looks like a mouse")
session.data_objects.put(output_filename, output_dir + '/' + output_filename) 

Load the schema called 'vcf' from the 'training' project.

In [ ]:
schema_file = get_mango_schema(session, "training", "vcf") 
vcf_schema = Schema(schema_file)

Inspect the fields requested by the metadata schema "vcf".

In [ ]:
print(vcf_schema)
vcf_schema.print_requirements("sample_id")

# or, if you want to inspect all fields at once:

for field in vcf_schema.fields:
    print(f"Requirements for field {field}:")
    vcf_schema.print_requirements(field)

Create a dictionary with the metadata you know that you could use to fill the VCF schema.
Note that "sample_id" should be a LIST with the values of `mgs.fastq.sample.sample_id` from your input files. So:
- Retrieve the sample_id and organism from the metadata on the input files
- Register the software used to generate the VCF file.

In [ ]:
objects = [session.data_objects.get(f"{x[0]}/{x[1]}") for x in paths]

In [ ]:
sample_ids = [x.metadata["mgs.fastq.sample.sample_id"].value for x in objects]
organism = objects[0].metadata["mgs.fastq.organism"].value
print(sample_ids)
print(organism)

In [ ]:
vcf_metadata = {
    "sample_id": sample_ids,
    "organism": organism,
    "genome_build": "hg38",
    "software": {
        "mapper": {
            "tool_name": "bwa",
            "version": "0.7.17-r1188"
        },
        "snp_caller": {
            "tool_name": "bcftools",
            "version": "1.19"
        }
    }
}

In [ ]:
vcf_metadata

Now that you have your dictionary, validate it and add the metadata to your uploaded output file.

In [ ]:
vcf_schema.validate(vcf_metadata)

In [ ]:
output_obj = session.data_objects.get(output_dir + '/' + output_filename) 
vcf_schema.apply(output_obj, vcf_metadata)

In [ ]:
output_obj.metadata.items()

In [ ]:
vcf_schema.from_avus(output_obj.metadata.items())

## CLEAN UP 
<div class="alert alert-block alert-warning">
    <font size=4><b>Do not forget to clean up your session!</b></font>
</div>

In [ ]:
# leave this cell at the end and running every time you are done
session.cleanup()